# Project Overview

## Objective

The objective of this project is to fine-tune a pre-trained BERT model for automatic news topic classification using the AG News dataset. The model learns to classify news articles into one of four categories:

- World
- Sports
- Business
- Science/Technology

This project demonstrates the use of transformer-based deep learning models for Natural Language Processing (NLP) tasks using the Hugging Face Transformers library.

In [9]:
!pip -q install transformers datasets evaluate accelerate

# Import Required Libraries


In [10]:
import torch
import numpy as np
import pandas as pd

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, f1_score

# Load the Dataset

In [11]:
# Load training data
train_df = pd.read_csv("train.csv", header=None)
# Load testing data
test_df = pd.read_csv("test.csv", header=None)
# Give meaningful column names
train_df.columns = ["Label", "Title", "Description"]
test_df.columns = ["Label", "Title", "Description"]
# Display first 5 rows
train_df.head()

,Label,Title,Description
0,Class Index,Title,Description
1,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
2,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
3,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
4,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...


# Explore the Dataset

In [12]:
# Combine title and description into one text column
train_df["Text"] = train_df["Title"] + " " + train_df["Description"]
test_df["Text"] = test_df["Title"] + " " + test_df["Description"]

# Keep only required columns
train_df = train_df[["Text", "Label"]]
test_df = test_df[["Text", "Label"]]

train_df.head()

,Text,Label
0,Title Description,Class Index
1,Wall St. Bears Claw Back Into the Black (Reute...,3
2,Carlyle Looks Toward Commercial Aerospace (Reu...,3
3,Oil and Economy Cloud Stocks' Outlook (Reuters...,3
4,Iraq Halts Oil Exports from Main Southern Pipe...,3


In [13]:
# Convert labels from 1-4 to 0-3

# Remove the header row that was loaded as data
train_df = train_df.iloc[1:].copy()
test_df = test_df.iloc[1:].copy()

# Convert 'Label' column to integer type and then subtract 1
train_df["Label"] = train_df["Label"].astype(int) - 1
test_df["Label"] = test_df["Label"].astype(int) - 1

train_df.head()

,Text,Label
1,Wall St. Bears Claw Back Into the Black (Reute...,2
2,Carlyle Looks Toward Commercial Aerospace (Reu...,2
3,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
4,Iraq Halts Oil Exports from Main Southern Pipe...,2
5,"Oil prices soar to all-time record, posing new...",2


# Load BERT Tokenizer

In [14]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the Text


In [15]:
train_encodings = tokenizer(
    train_df["Text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_df["Text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# Create PyTorch Dataset

In [16]:
class NewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, index):

        item = {}

        for key in self.encodings:
            item[key] = torch.tensor(self.encodings[key][index])

        item["labels"] = torch.tensor(self.labels[index])

        return item

    def __len__(self):
        return len(self.labels)

In [17]:
train_dataset = NewsDataset(
    train_encodings,
    train_df["Label"]
)

test_dataset = NewsDataset(
    test_encodings,
    test_df["Label"]
)

# Load Pre-trained BERT Model

In [18]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Define Evaluation Metrics

In [19]:
def compute_metrics(pred):

    predictions = np.argmax(pred.predictions, axis=1)

    accuracy = accuracy_score(pred.label_ids, predictions)

    f1 = f1_score(
        pred.label_ids,
        predictions,
        average="weighted"
    )

    return {
        "Accuracy": accuracy,
        "F1 Score": f1
    }

# Configure Training Parameters

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="bert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=1,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)

# Initialize Trainer

In [21]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 score
1,0.227373,0.216810,0.941842,0.941840


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=15000, training_loss=0.2873911730448405, metrics={'train_runtime': 3033.4712, 'train_samples_per_second': 39.559, 'train_steps_per_second': 4.945, 'total_flos': 7893473402880000.0, 'train_loss': 0.2873911730448405, 'epoch': 1.0})

# Evaluate Model Performance

In [23]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 score
0.227373,0.216810,1,0.941842,0.941840


{'eval_loss': 0.21681012213230133, 'eval_Accuracy': 0.9418421052631579, 'eval_F1 Score': 0.9418404678744499}


# Fine-tune the BERT Model

In [ ]:
text = "Apple launches a new AI-powered iPhone."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move inputs to the same device as the model (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {key: val.to(device) for key, val in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits).item()

labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

print("Predicted Category:", labels[prediction])

# Test the Model

In [25]:
model.save_pretrained("bert_news_classifier")

tokenizer.save_pretrained("bert_news_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_news_classifier/tokenizer_config.json',
 'bert_news_classifier/tokenizer.json')

# Save the Trained Model

In [26]:
from google.colab import files
import shutil

shutil.make_archive(
    "bert_news_classifier",
    "zip",
    "bert_news_classifier"
)

files.download("bert_news_classifier.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Conclusion

## Summary

In this project, a BERT-based transformer model was fine-tuned to classify news articles into four categories: World, Sports, Business, and Science/Technology.

### Steps Performed

- Loaded the AG News dataset from CSV files.
- Combined the title and description into a single text field.
- Tokenized the text using the BERT tokenizer.
- Fine-tuned the pre-trained `bert-base-uncased` model.
- Evaluated the model using Accuracy and F1 Score.
- Tested the model on a custom news headline.
- Saved the trained model for future use.

### Outcome

The fine-tuned BERT model successfully classified news articles into their respective categories. This project demonstrates how transfer learning with transformer models can achieve strong performance on text classification tasks while requiring only a relatively small amount of task-specific training.

### Skills Demonstrated

- Natural Language Processing (NLP)
- Transformer Models
- Transfer Learning
- Text Classification
- Model Evaluation
- Hugging Face Transformers
- Google Colab